# Aspectos de reviews PT-BR: treino do aluno no Kaggle

Treina o Qwen3-1.7B com LoRA para classificar 5 aspectos (entrega, produto, atendimento, preço, embalagem) em avaliações da Olist e avalia no conjunto ouro público. Os rótulos de treino vêm do Qwen3-8B, um professor aberto ([FMendonca/olist-aspectos-silver-qwen3-8b](https://huggingface.co/datasets/FMendonca/olist-aspectos-silver-qwen3-8b)).

O aluno treinado com os rótulos do Claude Haiku não é treinado aqui: a Política de Uso da Anthropic proíbe usar saídas do Claude para treinar modelos sem autorização prévia. Os resultados dele estão no repositório, como medida.

O mesmo código roda numa RX 7600 de 8 GB (ROCm) e aqui, numa T4. Repositório: https://github.com/fernandomcrsilva/review-aspects-ptbr

**Antes de rodar:** Settings → Accelerator → GPU T4 e Internet ligada.

In [ ]:
!git clone -q https://github.com/fernandomcrsilva/review-aspects-ptbr
%cd review-aspects-ptbr
# results/ traz os números da RX 7600; apaga para este notebook mostrar só o que rodar aqui
!rm -rf results
# o torchao do Kaggle é antigo demais para o peft atual, e o projeto não usa torchao
!pip uninstall -y -q torchao
!pip install -q 'peft>=0.17' 'transformers>=5.0'
!python src/get_data.py
# rótulos do professor aberto (Qwen3-8B), no mesmo formato que o train_lora.py espera
from huggingface_hub import snapshot_download
snapshot_download("FMendonca/olist-aspectos-silver-qwen3-8b", repo_type="dataset", local_dir="data/silver-qwen3-8b",
                  allow_patterns=["*.jsonl"])

## Baseline clássico: TF-IDF + regressão logística

In [ ]:
!python src/evaluate.py --name tfidf-qwen3-8b --tfidf data/silver-qwen3-8b

## Aluno sem ajuste (zero-shot, prompt completo)

In [ ]:
!python src/evaluate.py --name aluno-zero-shot

## LoRA nos rótulos do Qwen3-8B

Na T4 o modelo roda em fp32 (a T4 não tem bf16 nativo).

In [ ]:
!python src/train_lora.py --silver data/silver-qwen3-8b --out runs/aluno-qwen3-8b --batch 8 --accum 2

In [ ]:
!python src/evaluate.py --name aluno-qwen3-8b --adapter runs/aluno-qwen3-8b

## Resultados

In [ ]:
import json
for n in ['tfidf-qwen3-8b', 'aluno-zero-shot', 'aluno-qwen3-8b']:
    r = json.load(open(f'results/{n}.json'))
    print(f"{n:18} F1 {r['macro']:.3f} | pede ação {r['acao_recall']:.1%} | JSON válido {r['json_valido']:.0%} | {r['reviews_por_segundo']:.1f} avaliações/s")